In [1]:
# ═══ CELL 1 — Install (Versi Stabil untuk DETR)
!pip install fastapi "uvicorn[standard]" python-multipart pyngrok "transformers==4.38.2" -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.7/130.7 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 36.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.5.1 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.38.2 which is incompatible.


In [3]:
from google.colab import drive
import os

# ========================================================
# SOLUSI: Cek apakah Drive sudah ter-mount sebelumnya
# ========================================================
if not os.path.exists('/content/drive/MyDrive'):
    print('Menghubungkan ke Google Drive...')
    # Jika Anda ingin memaksa mount ulang, Anda juga bisa menggunakan:
    # drive.mount('/content/drive', force_remount=True)
    drive.mount('/content/drive', force_remount=True)
else:
    print('✅ Google Drive SUDAH terhubung sebelumnya!')

# Melanjutkan proses Anda
print('\nIsi folder MyDrive:')
try:
    print(os.listdir('/content/drive/MyDrive'))
except Exception as e:
    print(f"Gagal membaca isi folder: {e}")

Menghubungkan ke Google Drive...
Mounted at /content/drive

Isi folder MyDrive:
['Classroom', 'Biologi perlakuan pada kentang (1).docx', 'FLUIDA STATIS DINAMIS - CORONA Vincent Wijaya XI IPA (1).docx', 'Tugas bahasa indonesia teks eksplanasi (Banjir) Vincent W. XI IPA.docx', '201613.jpg', '201614.jpg', '201612.jpg', '204311.jpg', '214787.jpg', '214786.jpg', '221120.jpg', '221886.jpg', 'Cerpen Vincent W..docx', '227536.jpg', 'Teks ceramah Vincent Wijaya.pdf', 'Teks ceramah Vincent Wijaya (1).gdoc', 'Teks ceramah Vincent Wijaya.gdoc', 'Vincent Wijaya - Song Practice.gdoc', '239279.jpg', '239280.jpg', '239281.jpg', 'Tugas Mengidentifikasi Isi & bentuk penyampaian Ceramah yang Telah ditonton.docx', '243359.jpg', '243480.jpg', '243478.jpg', '243479.jpg', '250702.jpg', '251552.jpg', '251556.jpg', '251555.jpg', '251554.jpg', '251553.jpg', '253453.jpg', '253452.jpg', '253451.jpg', '253900.jpg', '253901.jpg', '254103.jpg', '254104.jpg', '257316 (1).jpg', 'IMG_20201122_172329.jpg', 'IMG_20201122

In [4]:
# ═══ Cek path yang benar ═══════════════════════════════════════
import json

MODEL_PATH = "/content/drive/MyDrive/detr_output_new_data_latih_2_70_20_10/checkpoints/epoch_50"

with open(f"{MODEL_PATH}/label_map.json") as f:
    print(json.load(f))

{'id2label': {'1': 'almond', '2': 'mete', '3': 'kerang', '4': 'kepiting', '5': 'telur', '6': 'ikan', '7': 'mie', '8': 'tahu', '9': 'tempe', '10': 'udang'}, 'label2id': {'almond': 1, 'mete': 2, 'kerang': 3, 'kepiting': 4, 'telur': 5, 'ikan': 6, 'mie': 7, 'tahu': 8, 'tempe': 9, 'udang': 10}, 'num_classes': 10}


In [5]:
%%writefile /content/api_server.py

import json, io
from fastapi import FastAPI, File, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from transformers import DetrImageProcessor, DetrForObjectDetection
import torch
from PIL import Image
import uvicorn

MODEL_PATH = "/content/drive/MyDrive/detr_output_new_data_latih_2_70_20_10/checkpoints/epoch_50"

# ── Load processor & model TANPA override id2label ────────────────
processor = DetrImageProcessor.from_pretrained(MODEL_PATH)
model     = DetrForObjectDetection.from_pretrained(
    MODEL_PATH,
    ignore_mismatched_sizes=True,
    # ✅ HAPUS num_labels, id2label, label2id — biarkan dari checkpoint
)
model.eval()

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# ── Ambil label langsung dari model.config (sumber kebenaran) ─────
ID2LABEL    = model.config.id2label          # {0:'background', 1:'almond', ...}
BACKGROUND  = {"background", "no_object"}    # label yang diabaikan
VALID_LABELS = {v for v in ID2LABEL.values() if v not in BACKGROUND}

print(f"✅ Model ready on {device}!")
print(f"📋 Valid labels: {sorted(VALID_LABELS)}", flush=True)

# ── Threshold — turunkan karena mAP model masih 20% ───────────────
CONFIDENCE_THRESHOLD = 0.5   # bisa turunkan ke 0.1 jika perlu

app = FastAPI()
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.post("/predict")
async def predict(image: UploadFile = File(...)):
    contents = await image.read()
    img      = Image.open(io.BytesIO(contents)).convert("RGB")
    orig_w, orig_h = img.size

    inputs = processor(images=img, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(**inputs)

    target_sizes = torch.tensor([[orig_h, orig_w]])
    results = processor.post_process_object_detection(
        outputs,
        threshold=CONFIDENCE_THRESHOLD,
        target_sizes=target_sizes,
    )[0]

    predictions = []
    for score, label_idx, box in zip(
        results["scores"], results["labels"], results["boxes"]
    ):
        label_name = ID2LABEL.get(int(label_idx), "background")

        # ✅ Skip background/no_object
        if label_name in BACKGROUND:
            continue

        x1, y1, x2, y2 = box.tolist()
        x1 = max(0.0, min(x1, orig_w))
        y1 = max(0.0, min(y1, orig_h))
        x2 = max(0.0, min(x2, orig_w))
        y2 = max(0.0, min(y2, orig_h))

        predictions.append({
            "label"      : label_name,
            "confidence" : round(float(score), 4),
            "is_allergen": True,
            "box"        : [
                round(x1 / orig_w, 4),
                round(y1 / orig_h, 4),
                round(x2 / orig_w, 4),
                round(y2 / orig_h, 4),
            ],
        })

    found = [p["label"] for p in predictions]
    print(f"📦 {len(predictions)} deteksi: {found}", flush=True)
    return {
        "predictions"     : predictions,
        "has_allergen"    : len(predictions) > 0,
        "allergen_summary": ", ".join(set(found)) if found else "Aman",
    }

@app.get("/health")
async def health():
    return {
        "status"       : "ok",
        "valid_labels" : sorted(VALID_LABELS),
        "threshold"    : CONFIDENCE_THRESHOLD,
        "device"       : device,
    }

if __name__ == "__main__":
    uvicorn.run("api_server:app", host="0.0.0.0", port=8000, reload=False)

Writing /content/api_server.py


In [6]:
# ═══ CELL 4 — Jalankan server + tunggu siap ══════════════════════
import subprocess, threading, time, socket

# Pastikan tidak ada server lama
subprocess.run(["pkill", "-f", "api_server.py"], capture_output=True)
time.sleep(2)

proc = subprocess.Popen(
    ["python", "/content/api_server.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
)

# Stream log realtime
def stream(pipe, label):
    for line in pipe:
        print(f"[{label}] {line}", end="")

threading.Thread(target=stream, args=(proc.stdout, "OUT"), daemon=True).start()
threading.Thread(target=stream, args=(proc.stderr, "ERR"), daemon=True).start()

# Tunggu port 8000 aktif (max 60 detik)
print("⏳ Menunggu server siap...")
ready = False
for i in range(60):
    time.sleep(1)
    if proc.poll() is not None:
        print("❌ Server crash! Lihat log [ERR] di atas.")
        break
    with socket.socket() as s:
        if s.connect_ex(("localhost", 8000)) == 0:
            print(f"✅ Server siap setelah {i+1} detik!")
            ready = True
            break

if not ready:
    print("⚠️ Server belum siap setelah 60 detik")

⏳ Menunggu server siap...
[OUT] ✅ Model ready on cpu!
[OUT] 📋 Valid labels: ['almond', 'ikan', 'kepiting', 'kerang', 'mete', 'mie', 'tahu', 'telur', 'tempe', 'udang']
[OUT] ✅ Model ready on cpu!
[OUT] 📋 Valid labels: ['almond', 'ikan', 'kepiting', 'kerang', 'mete', 'mie', 'tahu', 'telur', 'tempe', 'udang']
[ERR] INFO:     Started server process [10072]
[ERR] INFO:     Waiting for application startup.
[ERR] INFO:     Application startup complete.
[ERR] INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
✅ Server siap setelah 45 detik!


In [7]:
# ═══ CELL 5 — Connect ngrok + test (jalankan SETELAH cell 4 ✅) ══
from pyngrok import ngrok
import requests

# 1. Matikan sesi ngrok yang mungkin masih menggantung dari error sebelumnya
ngrok.kill()

# 2. Masukkan Authtoken Anda di sini
# GANTI teks "MASUKKAN_TOKEN_ANDA_DI_SINI" dengan token asli yang Anda salin dari dashboard
NGROK_AUTH_TOKEN = "3DIhuYTmEb3KkkUVW3uweb4psrG_7vQyV9H8j2u4bb2yUMbCa"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# 3. Buka koneksi ke port 8000
tunnel = ngrok.connect(8000)
public_url = tunnel.public_url

# 4. Cetak URL untuk disalin ke Flutter Anda
print(f"\n🚀 URL untuk Flutter:")
print(f"   static const String _apiUrl = '{public_url}/predict';")

print(f"🌐 URL: {public_url}")

r = requests.get(
    f"{public_url}/health",
    headers={"ngrok-skip-browser-warning": "true"},
    timeout=15,
)
print("✅ Health:", r.json())
print(f"\n📱 Salin ke Flutter _apiUrl:")
print(f"'{public_url}/predict'")


🚀 URL untuk Flutter:
   static const String _apiUrl = 'https://debit-trustable-flap.ngrok-free.dev/predict';
🌐 URL: https://debit-trustable-flap.ngrok-free.dev
[OUT] INFO:     34.19.8.52:0 - "GET /health HTTP/1.1" 200 OK
✅ Health: {'status': 'ok', 'valid_labels': ['almond', 'ikan', 'kepiting', 'kerang', 'mete', 'mie', 'tahu', 'telur', 'tempe', 'udang'], 'threshold': 0.5, 'device': 'cpu'}

📱 Salin ke Flutter _apiUrl:
'https://debit-trustable-flap.ngrok-free.dev/predict'


In [ ]:
# ═══ CELL DEBUG — Cek kondisi model secara langsung ══════════════
import torch, json
from PIL import Image
from transformers import DetrImageProcessor, DetrForObjectDetection

MODEL_PATH = "/content/drive/MyDrive/dhttps://debit-trustable-flap.ngrok-free.dev/predict/Detr_output/final_model/new_Dataset"
TEST_IMAGE_PATH = "/content/drive/MyDrive/Final_Dataset_Makanan/output_mie/test/images/mie_0028.jpg"

# ── 1. Load label map ──────────────────────────────────────────────
with open(f"{MODEL_PATH}/label_map.json") as f:
    label_data = json.load(f)

ID2LABEL      = {int(k): v for k, v in label_data["id2label"].items()}
ID2LABEL_FULL = {0: "no_object", **ID2LABEL, 11: "no_object"}
VALID_LABELS  = set(ID2LABEL.values())

print("📋 Label yang valid:", VALID_LABELS)
print("🗺️  ID2LABEL_FULL:", ID2LABEL_FULL)

# ── 2. Load model & processor ─────────────────────────────────────
processor = DetrImageProcessor.from_pretrained(MODEL_PATH)
model = DetrForObjectDetection.from_pretrained(MODEL_PATH, ignore_mismatched_sizes=True)
model.eval()

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print(f"\n✅ Model dimuat ke {device}")
print(f"📌 model.config.id2label: {model.config.id2label}")

# ── 3. Cek apakah model.config.id2label == ID2LABEL_FULL ──────────
print("\n🔍 Perbandingan label:")
for k in sorted(model.config.id2label.keys()):
    config_label = model.config.id2label[k]
    manual_label = ID2LABEL_FULL.get(k, "❌ TIDAK ADA")
    match = "✅" if config_label == manual_label else "❌ BEDA"
    print(f"  id={k}: config='{config_label}' | manual='{manual_label}' {match}")

# ── 4. Inferensi langsung (TANPA threshold dulu) ──────────────────
img = Image.open(TEST_IMAGE_PATH).convert("RGB")
orig_w, orig_h = img.size
print(f"\n🖼️  Gambar: {orig_w}x{orig_h}")

inputs = processor(images=img, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model(**inputs)

# Ambil SEMUA prediksi tanpa filter threshold
target_sizes = torch.tensor([[orig_h, orig_w]])
results_all = processor.post_process_object_detection(
    outputs,
    threshold=0.0,   # ← ambil semua 100 query DETR
    target_sizes=target_sizes,
)[0]

scores_all = results_all["scores"].cpu()
labels_all = results_all["labels"].cpu()

print(f"\n📊 Total query DETR: {len(scores_all)}")
print(f"📊 Max score       : {scores_all.max().item():.4f}")
print(f"📊 Min score       : {scores_all.min().item():.4f}")
print(f"📊 Mean score      : {scores_all.mean().item():.4f}")

# Top-20 prediksi
print(f"\n{'Rank':<5} {'Score':<10} {'LabelID':<10} {'LabelName':<20} {'Valid?'}")
print("─" * 60)
top_idx = scores_all.argsort(descending=True)[:20]
for rank, i in enumerate(top_idx):
    s   = float(scores_all[i])
    lid = int(labels_all[i])
    lname = model.config.id2label.get(lid, f"unknown_{lid}")
    valid = "✅" if lname in VALID_LABELS else "❌ (no_object/invalid)"
    print(f"{rank+1:<5} {s:<10.4f} {lid:<10} {lname:<20} {valid}")

# ── 5. Hitung berapa yang lolos tiap threshold ────────────────────
print("\n=== Sweep Threshold ===")
for thresh in [0.9, 0.7, 0.5, 0.3, 0.2, 0.1, 0.05, 0.01]:
    count_valid = sum(
        1 for i in range(len(scores_all))
        if float(scores_all[i]) > thresh
        and model.config.id2label.get(int(labels_all[i]), "") in VALID_LABELS
    )
    count_all = sum(1 for s in scores_all if float(s) > thresh)
    print(f"  thresh={thresh:.2f} → {count_valid} valid deteksi (dari {count_all} total)")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/dhttps://debit-trustable-flap.ngrok-free.dev/predict/Detr_output/final_model/new_Dataset/label_map.json'

In [ ]:
# ═══ Test dengan gambar asli dari dataset Anda ════════════════════
import requests
from PIL import Image
import io

# Ganti dengan path gambar test yang PASTI ada allergen
TEST_IMAGE_PATH = "/content/drive/MyDrive/Final_Dataset_Makanan/test_makanan/nasigoreng_00001.jpeg"
# Atau pakai gambar apapun yang mengandung udang/telur dll

with open(TEST_IMAGE_PATH, "rb") as f:
    r = requests.post(
        f"{public_url}/predict",
        files={"image": ("test.jpg", f, "image/jpeg")},
        headers={"ngrok-skip-browser-warning": "true"},
    )

print("Status:", r.status_code)
print("Response:", r.json())

[OUT] 📦 1 deteksi: ['telur']
[OUT] INFO:     35.196.23.42:0 - "POST /predict HTTP/1.1" 200 OK
Status: 200
Response: {'predictions': [{'label': 'telur', 'confidence': 0.9108, 'is_allergen': True, 'box': [0.0, 0.1417, 0.9844, 0.7672]}], 'has_allergen': True, 'allergen_summary': 'telur'}


In [ ]:
# ═══ Debug raw — lihat score TANPA threshold sama sekali ══════════
import torch, requests, io
from PIL import Image

img_url  = "https://upload.wikimedia.org/wikipedia/commons/thumb/5/5b/Mie_ayam.jpg/320px-Mie_ayam.jpg"
img_data = requests.get(img_url).content
img      = Image.open(io.BytesIO(img_data)).convert("RGB")

print(f"Ukuran gambar: {img.size}")

inputs = processor(images=img, return_tensors="pt")
with torch.no_grad():
    outputs = model(**inputs)

probs              = outputs.logits[0].softmax(-1)
scores, label_idxs = probs.max(-1)
sorted_idx         = scores.argsort(descending=True)

print(f"\n{'Rank':<5} {'Label':<15} {'Score':<10}")
print("-" * 32)
for rank, i in enumerate(sorted_idx[:15]):
    idx   = int(label_idxs[i])
    score = float(scores[i])
    name  = ID2LABEL_FULL.get(idx, f"class_{idx}")
    marker = " ✅" if name in VALID_LABELS else ""
    print(f"{rank+1:<5} {name:<15} {score:.4f}{marker}")

# Cek berapa query yang detect valid label di berbagai threshold
print("\n=== Jumlah deteksi per threshold ===")
for thresh in [0.9, 0.7, 0.5, 0.3, 0.1, 0.05]:
    count = sum(
        1 for i in range(len(scores))
        if float(scores[i]) > thresh
        and ID2LABEL_FULL.get(int(label_idxs[i]), "no_object") in VALID_LABELS
    )
    print(f"  threshold {thresh:.2f} → {count} deteksi")

UnidentifiedImageError: cannot identify image file <_io.BytesIO object at 0x7c7b6bbf8720>

In [ ]:
# ═══ Cek apakah weight classifier ter-load atau di-reinit ════════
import torch

w = model.class_labels_classifier.weight
print(f"Shape  : {w.shape}")           # harusnya [12, 256]
print(f"Mean   : {w.mean():.6f}")      # kalau ~0.0 → weight di-reinit!
print(f"Std    : {w.std():.6f}")       # kalau ~0.02 → reinit (Xavier init)
print(f"Min    : {w.min():.6f}")
print(f"Max    : {w.max():.6f}")
print(f"\nSample baris kelas 1 (almond):\n{w[1][:8]}")
print(f"Sample baris kelas 7 (mie):\n{w[7][:8]}")

NameError: name 'model' is not defined

In [ ]:
# ═══ Fix preprocessor_config.json langsung di Drive ══════════════
import json

config_path = f"{MODEL_PATH}/preprocessor_config.json"

new_config = {
    "_valid_processor_keys": [
        "images", "annotations", "return_segmentation_masks",
        "masks_path", "do_resize", "size", "resample",
        "do_rescale", "rescale_factor", "do_normalize",
        "do_convert_annotations", "image_mean", "image_std",
        "do_pad", "format", "return_tensors", "data_format",
        "input_data_format"
    ],
    "do_convert_annotations": True,
    "do_normalize"          : True,
    "do_pad"                : False,
    "do_rescale"            : True,
    "do_resize"             : True,       # ✅ sudah benar
    "format"                : "coco_detection",
    "image_mean"            : [0.485, 0.456, 0.406],
    "image_processor_type"  : "DetrImageProcessor",
    "image_std"             : [0.229, 0.224, 0.225],
    "resample"              : 2,
    "rescale_factor"        : 0.00392156862745098,

    # ✅ UBAH INI — dari longest/shortest edge ke height/width 640x640
    "size": {
        "height": 640,
        "width" : 640
    }
}

# Simpan
with open(config_path, "w") as f:
    json.dump(new_config, f, indent=2)

print("✅ preprocessor_config.json berhasil diupdate!")

# Verifikasi
with open(config_path) as f:
    saved = json.load(f)
print(f"   size    : {saved['size']}")
print(f"   do_resize: {saved['do_resize']}")

✅ preprocessor_config.json berhasil diupdate!
   size    : {'height': 640, 'width': 640}
   do_resize: True


In [ ]:
# ═══ Reload processor dari file yang sudah difix ═════════════════
from transformers import DetrImageProcessor

processor = DetrImageProcessor.from_pretrained(MODEL_PATH)

# Verifikasi
print(f"do_resize : {processor.do_resize}")
print(f"size      : {processor.size}")
# Harusnya: {'height': 640, 'width': 640}

do_resize : True
size      : {'height': 640, 'width': 640}


In [ ]:
# ═══ Test dengan gambar dari Drive Anda sendiri ═══════════════════
import torch, io
from PIL import Image

# Pakai salah satu gambar dari dataset training/test Anda
# Cari gambar yang ada di Drive
import os, glob

# Cari semua gambar jpg/png di folder dataset
search_paths = [
   "/content/drive/MyDrive/Final_Dataset_Makanan/output_mie/test/images/mie_0028.jpg",
    "/content/drive/MyDrive/Final_Dataset_Makanan/output_udang/test/images/udang_0030.jpg",
]

found_images = []
for pattern in search_paths:
    found_images.extend(glob.glob(pattern, recursive=True))

print(f"Gambar ditemukan: {len(found_images)}")
for img_path in found_images[:10]:
    print(f"  {img_path}")

Gambar ditemukan: 2
  /content/drive/MyDrive/Final_Dataset_Makanan/output_mie/test/images/mie_0028.jpg
  /content/drive/MyDrive/Final_Dataset_Makanan/output_udang/test/images/udang_0030.jpg


In [ ]:
import torch
from PIL import Image
from transformers import AutoModelForObjectDetection

# ==============================================================
# 1. MUAT MODEL KE DALAM MEMORI (Ini akan memperbaiki NameError)
# ==============================================================
# Gunakan path yang spasinya sudah kita hapus sebelumnya
MODEL_PATH = "/content/drive/MyDrive/detr_output/final_model/FINAL_MODEL_DETR/lr_1e-4_32_200/"

print("⏳ Memuat model dari Google Drive...")
# Inisialisasi variabel 'model'
model = AutoModelForObjectDetection.from_pretrained(MODEL_PATH)
model.eval() # Posisikan model ke mode evaluasi (bukan training)

# Pindahkan ke GPU jika tersedia agar inferensi jauh lebih cepat
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"✅ Model berhasil dimuat menggunakan {device}!\n")


# ==============================================================
# 2. KODE INFERENSI ANDA
# ==============================================================
# Ganti path ini dengan gambar yang mengandung allergen
TEST_IMAGE = found_images[0]
print(f"Test dengan: {TEST_IMAGE}")

img = Image.open(TEST_IMAGE).convert("RGB")
print(f"Ukuran: {img.size}")

# Memproses gambar menjadi tensor dan memindahkannya ke device yang sama dengan model
# (Asumsi variabel 'processor' sudah Anda definisikan di cell sebelumnya)
inputs = processor(images=img, return_tensors="pt").to(device)
print(f"Tensor shape: {inputs['pixel_values'].shape}")

# Eksekusi inferensi
with torch.no_grad():
    outputs = model(**inputs)

probs              = outputs.logits[0].softmax(-1)
scores, label_idxs = probs.max(-1)
sorted_idx         = scores.argsort(descending=True)

print(f"\n{'Rank':<5} {'Label':<15} {'Score':<10}")
print("-" * 32)
for rank, i in enumerate(sorted_idx[:10]):
    idx   = int(label_idxs[i])
    score = float(scores[i])
    name  = ID2LABEL_FULL.get(idx, f"class_{idx}")
    marker = " ✅" if name in VALID_LABELS else ""
    print(f"{rank+1:<5} {name:<15} {score:.4f}{marker}")

print("\n=== Deteksi per threshold ===")
for thresh in [0.9, 0.7, 0.5, 0.3, 0.1, 0.05]:
    count = sum(
        1 for i in range(len(scores))
        if float(scores[i]) > thresh
        and ID2LABEL_FULL.get(int(label_idxs[i]), "no_object") in VALID_LABELS
    )
    print(f"  threshold {thresh:.2f} → {count} deteksi")

In [ ]:
# ═══ Cek preprocessor_config — apakah size sesuai training? ═══════
import json
with open(f"{MODEL_PATH}/preprocessor_config.json") as f:
    cfg = json.load(f)
print(json.dumps(cfg, indent=2))